# EXP-006: Independent Crop Controller

An independently written baseline for Kaggriculture. It budgets existing cash, models future supply/demand, reserves worker tasks and transports terminal inventory.

Local tests beat the legacy repository baseline but lose to the public Shape v11 reference. This is a baseline/runtime audit, not a leaderboard-strength claim. No public opponent source, action tapes or weights are included in the submitted artifact.

Source and research: https://github.com/whzy3185/kagriculture/tree/research/round1-audit-20260907

The artifact is distributed under Apache-2.0, with game-rule attribution in NOTICE.txt. No credentials are included and this notebook does not submit to the competition.

In [ ]:
import importlib.metadata, subprocess, sys
if importlib.metadata.version("kaggle-environments") != "1.32.7":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle-environments==1.32.7"], check=True)
import hashlib, json, pathlib
from kaggle_environments import __version__
assert __version__ == "1.32.7"


In [ ]:
FILES = {'main.py': '"""Independent crop-economy controller; no public-agent source or tapes.\n\nGame constants and price equations follow Kaggle\'s Apache-2.0 environment.\nPolicy, budgeting, task assignment and terminal transport are independently written.\n"""\n\nimport math\n\nDEMAND_AWARE = True\nEXPAND_LAND = True\nADAPTIVE_LABOR = True\nTERMINAL_CASH = True\n\n# seed, first harvest, unfertilized peak age, interval, yield, base, bonus start\nCROPS = {\n    "WHEAT": (10, 2, 4, 0, 4, 25, 2),\n    "CARROT": (20, 2, 3, 0, 3, 35, 2),\n    "TOMATO": (50, 8, 11, 1, 4, 60, 0),\n    "STRAWBERRY": (100, 10, 16, 2, 4, 120, 0),\n    "MELON": (80, 10, 10, 0, 6, 250, 6),\n}\nSHOPS = {\n    "BAKERY": ("EGG", "WHEAT"),\n    "PIZZA_SHOP": ("MILK", "TOMATO", "WHEAT"),\n    "BRUNCH_SPOT": ("EGG", "WHEAT", "STRAWBERRY"),\n    "YARN_STORE": ("WOOL",),\n    "ICE_CREAM_SHOP": ("STRAWBERRY", "MILK", "WHEAT"),\n    "PET_CAFE": ("CARROT",),\n    "SMOOTHIE_SHOP": ("STRAWBERRY", "MILK"),\n    "FARMERS_MARKET": ("WHEAT", "CARROT", "TOMATO", "STRAWBERRY"),\n}\n# base, throughput, scarcity shape/target, surplus shape/target\nPRICES = {\n    "WHEAT": (25, 400, "sqrt", .8, "log", .2),\n    "CARROT": (35, 450, "hinge", 1., "sqrt", .7),\n    "TOMATO": (60, 200, "hinge", .4, "sqrt", .6),\n    "STRAWBERRY": (120, 100, "sqrt", .7, "linear", 1.6),\n    "MELON": (250, 300, "log", .2, "sq", 3.6),\n    "EGG": (50, 332, "hinge", .4, "log", .2),\n    "MILK": (160, 122, "sqrt", .6, "linear", 1.6),\n    "WOOL": (200, 105, "log", .2, "sq", 3.2),\n    "FERTILIZER": (100, 200, "linear", .4, "linear", .4),\n}\n\n\ndef _shape(kind, x, throughput):\n    x = max(0., x)\n    if kind == "sqrt":\n        return math.sqrt(x)\n    if kind == "log":\n        return math.log1p(x)\n    if kind == "log10":\n        return math.log10(1. + x)\n    if kind == "sq":\n        return x * x\n    if kind == "hinge":\n        u = x / throughput\n        return u + 8. * max(0., u - 1.) ** 2\n    return x\n\n\ndef price(item, inventory, params=None):\n    base, throughput, below, bt, above, at = PRICES[item]\n    p = (params or {}).get(item, {})\n    base, throughput = p.get("base", base), p.get("T", throughput)\n    equilibrium = p.get("I0", 10000)\n    low = inventory < equilibrium\n    kind = p.get("below_func", below) if low else p.get("above_func", above)\n    target = p.get("below_target", bt) if low else p.get("above_target", at)\n    move = target * base * _shape(kind, abs(inventory - equilibrium), throughput) / _shape(kind, throughput, throughput)\n    return max(1, round(base + move if low else base - move))\n\n\ndef _distance(a, b):\n    return abs(a[0] - b[0]) + abs(a[1] - b[1])\n\n\ndef _move(a, b):\n    if a[0] != b[0]:\n        return ["EAST" if a[0] < b[0] else "WEST"]\n    if a[1] != b[1]:\n        return ["SOUTH" if a[1] < b[1] else "NORTH"]\n    return ["PASS"]\n\n\ndef _fib(n):\n    a, b = 1, 1\n    for _ in range(n):\n        a, b = b, a + b\n    return a\n\n\ndef _demand(item, shops, day, horizon, config):\n    interval = config.get("townShopUnlockInterval", 3)\n    ticks = config.get("turnsPerDay", 24) / config.get("townShopSellInterval", 4)\n    center = config.get("turnsPerDay", 24) / config.get("townCenterSellInterval", 24)\n    rate = sum((2 if len(SHOPS[s]) == 1 else 1) for s in shops if s in SHOPS and item in SHOPS[s]) * ticks\n    expected_new = sum((2 if len(v) == 1 else 1) for v in SHOPS.values() if item in v) * ticks / len(SHOPS)\n    count, result = len(shops), 0.\n    for offset in range(1, horizon + 1):\n        if (day + offset) % interval == 0 and count < 8:\n            rate += expected_new\n            count += 1\n        result += rate + center\n    return result\n\n\ndef _supply(crop, farms, day, horizon):\n    _, first, peak, interval, units, _, _ = CROPS[crop]\n    result = 0\n    for farm in farms:\n        for row in farm.get("tiles", []):\n            for tile in row:\n                if not isinstance(tile, dict) or tile.get("crop") != crop:\n                    continue\n                age = day - tile["planted_day"]\n                if interval:\n                    result += tile.get("yield_units", 0)\n                    result += sum(age < first + k * interval <= age + horizon for k in range(4))\n                elif age + horizon >= first:\n                    result += max(tile.get("yield_units", 0), units)\n    return result\n\n\ndef crop_values(obs, config, last_day):\n    day = obs["day"]\n    inventory = obs["market"]["inventory"]\n    params = obs["market"].get("params", {})\n    shops = obs.get("town", {}).get("unlocked_shops", [])\n    values = {}\n    for crop, (cost, first, peak, interval, maximum, base, bonus) in CROPS.items():\n        horizon = min(peak, last_day - 1 - day)\n        if horizon < first:\n            continue\n        units = sum(first + i * interval <= horizon for i in range(4)) if interval else min(maximum, 1 + max(0, horizon - bonus + 1))\n        if DEMAND_AWARE:\n            future_inventory = inventory[crop] + _supply(crop, obs["farms"], day, horizon) + units\n            future_inventory -= _demand(crop, shops, day, horizon, config)\n            expected_price = min(base * 3., price(crop, future_inventory, params)) * .9\n        else:\n            expected_price = base\n        values[crop] = (units * expected_price - cost) / (horizon + 1)\n    return values\n\n\ndef agent(obs, configuration=None):\n    farms = obs.get("farms") or []\n    player = int(obs.get("player", 0))\n    if not farms or not 0 <= player < len(farms):\n        return {"farmer": ["PASS"], "hands": [], "market": []}\n    farm = farms[player]\n    tiles = farm.get("tiles") or []\n    if not tiles:\n        return {"farmer": ["PASS"], "hands": [], "market": []}\n    config = configuration or {}\n    tpd, total = config.get("turnsPerDay", 24), config.get("episodeSteps", 720)\n    day, hour = int(obs.get("day", 0)), int(obs.get("hour", 0))\n    turn = day * tpd + hour\n    last_day, turns_left = (total - 2) // tpd, total - 1 - turn\n    final_day = TERMINAL_CASH and day == last_day\n    private = obs.get("private") or {}\n    seeds = dict(private.get("seeds") or {})\n    carried = [dict(i) for i in private.get("inventories", [])]\n    positions = [tuple(farm.get("farmer", [0, 0])), *map(tuple, farm.get("hands", []))]\n    while len(carried) < len(positions):\n        carried.append({})\n    shed = dict(private.get("shed") or {})\n    capacity = config.get("shedCapacity", 100)\n    cargo = sum(sum(inv.values()) for inv in carried)\n    half = len(tiles) // 2\n    access = [(half - 1, half - 1), (half, half - 1), (half - 1, half), (half, half)]\n    quotes = (obs.get("market") or {}).get("prices", {})\n    market_obs = dict(obs.get("market") or {})\n    market_obs.setdefault("inventory", {k: 10000 for k in PRICES})\n    model_obs = dict(obs, market=market_obs, day=day)\n    values = crop_values(model_obs, config, last_day)\n    plantable = hour <= tpd - 2 and bool(values)\n    active, empty = 0, 0\n    jobs = {}\n    for y, row in enumerate(tiles):\n        for x, tile in enumerate(row):\n            pos = (x, y)\n            if tile is None:\n                empty += 1\n                if plantable:\n                    jobs[pos] = ("PLANT", 90.)\n            elif isinstance(tile, dict) and tile.get("kind") == "WEED":\n                if plantable:\n                    jobs[pos] = ("DIG", 50.)\n            elif isinstance(tile, dict) and tile.get("crop") in CROPS:\n                active += 1\n                crop = tile["crop"]\n                _, first, peak, interval, _, base, bonus = CROPS[crop]\n                age = day - tile["planted_day"]\n                watered = bool(tile.get("watered_today"))\n                yield_units = int(tile.get("yield_units", 0))\n                bonus_water = not interval and bonus <= age <= (12 if crop == "MELON" else peak)\n                needs_water = not watered and (tile.get("consecutive_unwatered", 0) >= 1 or bonus_water)\n                ready = yield_units > 0 and age >= first and (interval or age >= peak or day >= last_day - 1)\n                if needs_water and not (final_day and not bonus_water):\n                    urgency = 4000. if tile.get("consecutive_unwatered", 0) >= 1 else 1500.\n                    jobs[pos] = ("WATER", urgency)\n                elif ready:\n                    weight = 500. + min(1500., yield_units * quotes.get(crop, base))\n                    jobs[pos] = ("HARVEST", weight)\n\n    actions = [None] * len(positions)\n    worked, destinations = set(), set()\n\n    def eligible(pos, origin):\n        if pos in worked:\n            return False\n        op, _ = jobs[pos]\n        if op == "PLANT" and not any(seeds.get(c, 0) > 0 for c in values):\n            return False\n        if op == "HARVEST":\n            units = tiles[pos[1]][pos[0]]["yield_units"]\n            if cargo + units > capacity:\n                return False\n            if final_day:\n                return _distance(origin, pos) + 1 + min(_distance(pos, a) for a in access) + 1 <= turns_left\n        if final_day and op == "WATER":\n            return _distance(origin, pos) + 2 + min(_distance(pos, a) for a in access) + 1 <= turns_left\n        return True\n\n    # Local work is assigned before travel, with separate completed-tile reservations.\n    for i, pos in enumerate(positions):\n        inv = carried[i]\n        held = sum(inv.values())\n        nearest_shed = min(access, key=lambda a: (_distance(pos, a), a))\n        returning = held > 0 and (final_day or cargo > 65 or held >= 12)\n        if held and pos in access and (returning or hour >= tpd - 4):\n            room = max(0, capacity - sum(shed.values()))\n            if room >= held:\n                actions[i] = ["DROP"]\n                for item, quantity in inv.items():\n                    shed[item] = shed.get(item, 0) + quantity\n                cargo -= held\n                continue\n            if room:\n                item = max(inv, key=lambda item: (quotes.get(item, 0), item))\n                amount = min(room, inv[item])\n                actions[i] = ["PLACE", item, amount]\n                shed[item] = shed.get(item, 0) + amount\n                cargo -= amount\n                continue\n        if returning and pos not in access:\n            actions[i] = _move(pos, nearest_shed)\n            continue\n        if pos not in jobs or not eligible(pos, pos):\n            continue\n        op, _ = jobs[pos]\n        if op == "PLANT":\n            choices = [c for c in values if seeds.get(c, 0) > 0]\n            crop = max(choices, key=lambda c: (values[c], c))\n            seeds[crop] -= 1\n            actions[i] = ["PLANT", crop]\n        else:\n            actions[i] = [op]\n            if op == "HARVEST":\n                cargo += tiles[pos[1]][pos[0]]["yield_units"]\n        worked.add(pos)\n\n    for i, pos in enumerate(positions):\n        if actions[i] is not None:\n            continue\n        possible = [p for p in jobs if p not in destinations and p != pos and eligible(p, pos)]\n        if possible:\n            target = max(possible, key=lambda p: (jobs[p][1] / (1 + _distance(pos, p)), -p[1], -p[0]))\n            actions[i] = _move(pos, target)\n            destinations.add(target)\n        else:\n            actions[i] = ["PASS"]\n\n    limit = config.get("maxMarketOrdersPerTurn", 10)\n    market = [["SELL", item, int(q)] for item, q in sorted(shed.items()) if item in PRICES and q > 0][:limit]\n    budget = float(farm.get("money", 0))\n    # Purchases spend existing cash only. Unknown simultaneous sale receipts are not borrowed.\n    if hour <= 2 and (active or empty and plantable or final_day and cargo):\n        target = min(10, max(3, math.ceil((active + min(empty, 10) if plantable else active) / 7))) if ADAPTIVE_LABOR else min(4, active // 5)\n        hires_today = int(farm.get("hires_today", len(farm.get("hands", []))))\n        for count in range(max(0, target - len(farm.get("hands", [])))):\n            cost = _fib(hires_today + count) * config.get("farmHandCostMult", 1)\n            if budget < cost + 20 or len(market) >= limit:\n                break\n            market.append(["HIRE"])\n            budget -= cost\n    quadrants = len(farm.get("unlocked_quadrants", ["NW"]))\n    unlocked = sum(tile != "LOCKED" for row in tiles for tile in row)\n    if EXPAND_LAND and quadrants < 3 and day <= last_day - 6 and active >= unlocked * .75:\n        cost = (1000, 2000, 4000)[quadrants - 1]\n        if budget >= cost + 800 and len(market) < limit:\n            market.append(["BUY_LAND"])\n            budget -= cost\n    if plantable and empty and values and len(market) < limit:\n        best = max(values, key=lambda c: (values[c], c))\n        if values[best] > 0:\n            # Purchases arrive after this turn\'s planting; replenish the remaining visible stock.\n            need = max(0, min(6, empty) - seeds.get(best, 0))\n            amount = min(need, max(0, int((budget - 80) // CROPS[best][0])))\n            if amount:\n                market.append(["BUY_SEED", best, amount])\n    return {"farmer": actions[0], "hands": actions[1:], "market": market}\n', 'agents/candidates/exp006_independent.py': '"""Independent crop-economy controller; no public-agent source or tapes.\n\nGame constants and price equations follow Kaggle\'s Apache-2.0 environment.\nPolicy, budgeting, task assignment and terminal transport are independently written.\n"""\n\nimport math\n\nDEMAND_AWARE = True\nEXPAND_LAND = True\nADAPTIVE_LABOR = True\nTERMINAL_CASH = True\n\n# seed, first harvest, unfertilized peak age, interval, yield, base, bonus start\nCROPS = {\n    "WHEAT": (10, 2, 4, 0, 4, 25, 2),\n    "CARROT": (20, 2, 3, 0, 3, 35, 2),\n    "TOMATO": (50, 8, 11, 1, 4, 60, 0),\n    "STRAWBERRY": (100, 10, 16, 2, 4, 120, 0),\n    "MELON": (80, 10, 10, 0, 6, 250, 6),\n}\nSHOPS = {\n    "BAKERY": ("EGG", "WHEAT"),\n    "PIZZA_SHOP": ("MILK", "TOMATO", "WHEAT"),\n    "BRUNCH_SPOT": ("EGG", "WHEAT", "STRAWBERRY"),\n    "YARN_STORE": ("WOOL",),\n    "ICE_CREAM_SHOP": ("STRAWBERRY", "MILK", "WHEAT"),\n    "PET_CAFE": ("CARROT",),\n    "SMOOTHIE_SHOP": ("STRAWBERRY", "MILK"),\n    "FARMERS_MARKET": ("WHEAT", "CARROT", "TOMATO", "STRAWBERRY"),\n}\n# base, throughput, scarcity shape/target, surplus shape/target\nPRICES = {\n    "WHEAT": (25, 400, "sqrt", .8, "log", .2),\n    "CARROT": (35, 450, "hinge", 1., "sqrt", .7),\n    "TOMATO": (60, 200, "hinge", .4, "sqrt", .6),\n    "STRAWBERRY": (120, 100, "sqrt", .7, "linear", 1.6),\n    "MELON": (250, 300, "log", .2, "sq", 3.6),\n    "EGG": (50, 332, "hinge", .4, "log", .2),\n    "MILK": (160, 122, "sqrt", .6, "linear", 1.6),\n    "WOOL": (200, 105, "log", .2, "sq", 3.2),\n    "FERTILIZER": (100, 200, "linear", .4, "linear", .4),\n}\n\n\ndef _shape(kind, x, throughput):\n    x = max(0., x)\n    if kind == "sqrt":\n        return math.sqrt(x)\n    if kind == "log":\n        return math.log1p(x)\n    if kind == "log10":\n        return math.log10(1. + x)\n    if kind == "sq":\n        return x * x\n    if kind == "hinge":\n        u = x / throughput\n        return u + 8. * max(0., u - 1.) ** 2\n    return x\n\n\ndef price(item, inventory, params=None):\n    base, throughput, below, bt, above, at = PRICES[item]\n    p = (params or {}).get(item, {})\n    base, throughput = p.get("base", base), p.get("T", throughput)\n    equilibrium = p.get("I0", 10000)\n    low = inventory < equilibrium\n    kind = p.get("below_func", below) if low else p.get("above_func", above)\n    target = p.get("below_target", bt) if low else p.get("above_target", at)\n    move = target * base * _shape(kind, abs(inventory - equilibrium), throughput) / _shape(kind, throughput, throughput)\n    return max(1, round(base + move if low else base - move))\n\n\ndef _distance(a, b):\n    return abs(a[0] - b[0]) + abs(a[1] - b[1])\n\n\ndef _move(a, b):\n    if a[0] != b[0]:\n        return ["EAST" if a[0] < b[0] else "WEST"]\n    if a[1] != b[1]:\n        return ["SOUTH" if a[1] < b[1] else "NORTH"]\n    return ["PASS"]\n\n\ndef _fib(n):\n    a, b = 1, 1\n    for _ in range(n):\n        a, b = b, a + b\n    return a\n\n\ndef _demand(item, shops, day, horizon, config):\n    interval = config.get("townShopUnlockInterval", 3)\n    ticks = config.get("turnsPerDay", 24) / config.get("townShopSellInterval", 4)\n    center = config.get("turnsPerDay", 24) / config.get("townCenterSellInterval", 24)\n    rate = sum((2 if len(SHOPS[s]) == 1 else 1) for s in shops if s in SHOPS and item in SHOPS[s]) * ticks\n    expected_new = sum((2 if len(v) == 1 else 1) for v in SHOPS.values() if item in v) * ticks / len(SHOPS)\n    count, result = len(shops), 0.\n    for offset in range(1, horizon + 1):\n        if (day + offset) % interval == 0 and count < 8:\n            rate += expected_new\n            count += 1\n        result += rate + center\n    return result\n\n\ndef _supply(crop, farms, day, horizon):\n    _, first, peak, interval, units, _, _ = CROPS[crop]\n    result = 0\n    for farm in farms:\n        for row in farm.get("tiles", []):\n            for tile in row:\n                if not isinstance(tile, dict) or tile.get("crop") != crop:\n                    continue\n                age = day - tile["planted_day"]\n                if interval:\n                    result += tile.get("yield_units", 0)\n                    result += sum(age < first + k * interval <= age + horizon for k in range(4))\n                elif age + horizon >= first:\n                    result += max(tile.get("yield_units", 0), units)\n    return result\n\n\ndef crop_values(obs, config, last_day):\n    day = obs["day"]\n    inventory = obs["market"]["inventory"]\n    params = obs["market"].get("params", {})\n    shops = obs.get("town", {}).get("unlocked_shops", [])\n    values = {}\n    for crop, (cost, first, peak, interval, maximum, base, bonus) in CROPS.items():\n        horizon = min(peak, last_day - 1 - day)\n        if horizon < first:\n            continue\n        units = sum(first + i * interval <= horizon for i in range(4)) if interval else min(maximum, 1 + max(0, horizon - bonus + 1))\n        if DEMAND_AWARE:\n            future_inventory = inventory[crop] + _supply(crop, obs["farms"], day, horizon) + units\n            future_inventory -= _demand(crop, shops, day, horizon, config)\n            expected_price = min(base * 3., price(crop, future_inventory, params)) * .9\n        else:\n            expected_price = base\n        values[crop] = (units * expected_price - cost) / (horizon + 1)\n    return values\n\n\ndef agent(obs, configuration=None):\n    farms = obs.get("farms") or []\n    player = int(obs.get("player", 0))\n    if not farms or not 0 <= player < len(farms):\n        return {"farmer": ["PASS"], "hands": [], "market": []}\n    farm = farms[player]\n    tiles = farm.get("tiles") or []\n    if not tiles:\n        return {"farmer": ["PASS"], "hands": [], "market": []}\n    config = configuration or {}\n    tpd, total = config.get("turnsPerDay", 24), config.get("episodeSteps", 720)\n    day, hour = int(obs.get("day", 0)), int(obs.get("hour", 0))\n    turn = day * tpd + hour\n    last_day, turns_left = (total - 2) // tpd, total - 1 - turn\n    final_day = TERMINAL_CASH and day == last_day\n    private = obs.get("private") or {}\n    seeds = dict(private.get("seeds") or {})\n    carried = [dict(i) for i in private.get("inventories", [])]\n    positions = [tuple(farm.get("farmer", [0, 0])), *map(tuple, farm.get("hands", []))]\n    while len(carried) < len(positions):\n        carried.append({})\n    shed = dict(private.get("shed") or {})\n    capacity = config.get("shedCapacity", 100)\n    cargo = sum(sum(inv.values()) for inv in carried)\n    half = len(tiles) // 2\n    access = [(half - 1, half - 1), (half, half - 1), (half - 1, half), (half, half)]\n    quotes = (obs.get("market") or {}).get("prices", {})\n    market_obs = dict(obs.get("market") or {})\n    market_obs.setdefault("inventory", {k: 10000 for k in PRICES})\n    model_obs = dict(obs, market=market_obs, day=day)\n    values = crop_values(model_obs, config, last_day)\n    plantable = hour <= tpd - 2 and bool(values)\n    active, empty = 0, 0\n    jobs = {}\n    for y, row in enumerate(tiles):\n        for x, tile in enumerate(row):\n            pos = (x, y)\n            if tile is None:\n                empty += 1\n                if plantable:\n                    jobs[pos] = ("PLANT", 90.)\n            elif isinstance(tile, dict) and tile.get("kind") == "WEED":\n                if plantable:\n                    jobs[pos] = ("DIG", 50.)\n            elif isinstance(tile, dict) and tile.get("crop") in CROPS:\n                active += 1\n                crop = tile["crop"]\n                _, first, peak, interval, _, base, bonus = CROPS[crop]\n                age = day - tile["planted_day"]\n                watered = bool(tile.get("watered_today"))\n                yield_units = int(tile.get("yield_units", 0))\n                bonus_water = not interval and bonus <= age <= (12 if crop == "MELON" else peak)\n                needs_water = not watered and (tile.get("consecutive_unwatered", 0) >= 1 or bonus_water)\n                ready = yield_units > 0 and age >= first and (interval or age >= peak or day >= last_day - 1)\n                if needs_water and not (final_day and not bonus_water):\n                    urgency = 4000. if tile.get("consecutive_unwatered", 0) >= 1 else 1500.\n                    jobs[pos] = ("WATER", urgency)\n                elif ready:\n                    weight = 500. + min(1500., yield_units * quotes.get(crop, base))\n                    jobs[pos] = ("HARVEST", weight)\n\n    actions = [None] * len(positions)\n    worked, destinations = set(), set()\n\n    def eligible(pos, origin):\n        if pos in worked:\n            return False\n        op, _ = jobs[pos]\n        if op == "PLANT" and not any(seeds.get(c, 0) > 0 for c in values):\n            return False\n        if op == "HARVEST":\n            units = tiles[pos[1]][pos[0]]["yield_units"]\n            if cargo + units > capacity:\n                return False\n            if final_day:\n                return _distance(origin, pos) + 1 + min(_distance(pos, a) for a in access) + 1 <= turns_left\n        if final_day and op == "WATER":\n            return _distance(origin, pos) + 2 + min(_distance(pos, a) for a in access) + 1 <= turns_left\n        return True\n\n    # Local work is assigned before travel, with separate completed-tile reservations.\n    for i, pos in enumerate(positions):\n        inv = carried[i]\n        held = sum(inv.values())\n        nearest_shed = min(access, key=lambda a: (_distance(pos, a), a))\n        returning = held > 0 and (final_day or cargo > 65 or held >= 12)\n        if held and pos in access and (returning or hour >= tpd - 4):\n            room = max(0, capacity - sum(shed.values()))\n            if room >= held:\n                actions[i] = ["DROP"]\n                for item, quantity in inv.items():\n                    shed[item] = shed.get(item, 0) + quantity\n                cargo -= held\n                continue\n            if room:\n                item = max(inv, key=lambda item: (quotes.get(item, 0), item))\n                amount = min(room, inv[item])\n                actions[i] = ["PLACE", item, amount]\n                shed[item] = shed.get(item, 0) + amount\n                cargo -= amount\n                continue\n        if returning and pos not in access:\n            actions[i] = _move(pos, nearest_shed)\n            continue\n        if pos not in jobs or not eligible(pos, pos):\n            continue\n        op, _ = jobs[pos]\n        if op == "PLANT":\n            choices = [c for c in values if seeds.get(c, 0) > 0]\n            crop = max(choices, key=lambda c: (values[c], c))\n            seeds[crop] -= 1\n            actions[i] = ["PLANT", crop]\n        else:\n            actions[i] = [op]\n            if op == "HARVEST":\n                cargo += tiles[pos[1]][pos[0]]["yield_units"]\n        worked.add(pos)\n\n    for i, pos in enumerate(positions):\n        if actions[i] is not None:\n            continue\n        possible = [p for p in jobs if p not in destinations and p != pos and eligible(p, pos)]\n        if possible:\n            target = max(possible, key=lambda p: (jobs[p][1] / (1 + _distance(pos, p)), -p[1], -p[0]))\n            actions[i] = _move(pos, target)\n            destinations.add(target)\n        else:\n            actions[i] = ["PASS"]\n\n    limit = config.get("maxMarketOrdersPerTurn", 10)\n    market = [["SELL", item, int(q)] for item, q in sorted(shed.items()) if item in PRICES and q > 0][:limit]\n    budget = float(farm.get("money", 0))\n    # Purchases spend existing cash only. Unknown simultaneous sale receipts are not borrowed.\n    if hour <= 2 and (active or empty and plantable or final_day and cargo):\n        target = min(10, max(3, math.ceil((active + min(empty, 10) if plantable else active) / 7))) if ADAPTIVE_LABOR else min(4, active // 5)\n        hires_today = int(farm.get("hires_today", len(farm.get("hands", []))))\n        for count in range(max(0, target - len(farm.get("hands", [])))):\n            cost = _fib(hires_today + count) * config.get("farmHandCostMult", 1)\n            if budget < cost + 20 or len(market) >= limit:\n                break\n            market.append(["HIRE"])\n            budget -= cost\n    quadrants = len(farm.get("unlocked_quadrants", ["NW"]))\n    unlocked = sum(tile != "LOCKED" for row in tiles for tile in row)\n    if EXPAND_LAND and quadrants < 3 and day <= last_day - 6 and active >= unlocked * .75:\n        cost = (1000, 2000, 4000)[quadrants - 1]\n        if budget >= cost + 800 and len(market) < limit:\n            market.append(["BUY_LAND"])\n            budget -= cost\n    if plantable and empty and values and len(market) < limit:\n        best = max(values, key=lambda c: (values[c], c))\n        if values[best] > 0:\n            # Purchases arrive after this turn\'s planting; replenish the remaining visible stock.\n            need = max(0, min(6, empty) - seeds.get(best, 0))\n            amount = min(need, max(0, int((budget - 80) // CROPS[best][0])))\n            if amount:\n                market.append(["BUY_SEED", best, amount])\n    return {"farmer": actions[0], "hands": actions[1:], "market": market}\n', 'legacy_parent.py': '"""Self-contained Kaggriculture baseline agent.\n\nDesign goals:\n- valid JSON-safe actions\n- deterministic decisions\n- no dependency on project-local modules\n- conservative inventory accounting for simultaneous PLANT actions\n- simple crop / labor / market policy that is easy to benchmark and replace\n"""\n\nfrom __future__ import annotations\n\nfrom typing import Any, Dict, Iterable, List, Optional, Sequence, Set, Tuple\n\nCROP_RULES = {\n    "WHEAT": {"seed_cost": 10, "max_yield_day": 4, "base_price": 25, "last_plant_day": 25},\n    "MELON": {"seed_cost": 80, "max_yield_day": 10, "base_price": 250, "last_plant_day": 18},\n}\n\n# Start with a robust short-cycle crop, switch to melons once cash is available,\n# then return to wheat near the end so capital is not stranded.\nMELON_START_CASH = 500\nTARGET_SEED_BUFFER = 8\nMAX_DAILY_HIRES = 4\nSELL_PRICE_RATIO = 0.90\nLATE_LIQUIDATION_DAY = 28\n\nCoord = Tuple[int, int]\nAction = List[Any]\n\n\ndef _prices(obs: Dict[str, Any]) -> Dict[str, float]:\n    """Support both public observation layouts seen in notebooks/environment."""\n    market = obs.get("market") or {}\n    prices = market.get("prices")\n    if isinstance(prices, dict):\n        return prices\n    direct = obs.get("prices")\n    return direct if isinstance(direct, dict) else {}\n\n\ndef _step_toward(start: Coord, target: Coord) -> Optional[str]:\n    sx, sy = start\n    tx, ty = target\n    if sx > tx:\n        return "WEST"\n    if sx < tx:\n        return "EAST"\n    if sy > ty:\n        return "NORTH"\n    if sy < ty:\n        return "SOUTH"\n    return None\n\n\ndef _tile_kind(tile: Any) -> Optional[str]:\n    if not isinstance(tile, dict):\n        return None\n    return tile.get("kind")\n\n\ndef _crop_name(tile: Any) -> Optional[str]:\n    if not isinstance(tile, dict):\n        return None\n    return tile.get("crop") or tile.get("seed")\n\n\ndef _is_unlocked_empty(tile: Any) -> bool:\n    return tile is None\n\n\ndef _is_weed(tile: Any) -> bool:\n    return isinstance(tile, dict) and tile.get("kind") == "WEED"\n\n\ndef _is_plant(tile: Any) -> bool:\n    return isinstance(tile, dict) and tile.get("kind") == "PLANT"\n\n\ndef _harvest_ready(tile: Any, day: int) -> bool:\n    if not _is_plant(tile):\n        return False\n\n    crop = _crop_name(tile)\n    planted_day = tile.get("planted_day")\n    yield_units = tile.get("yield_units", 0) or 0\n\n    if yield_units <= 0:\n        return False\n\n    # For crops we actively plant, wait for the configured peak day.\n    if crop in CROP_RULES and planted_day is not None:\n        age = day - int(planted_day)\n        return age >= int(CROP_RULES[crop]["max_yield_day"])\n\n    # Unknown/public crops: if the engine reports yield, harvesting is safer\n    # than ignoring mature produce forever.\n    return True\n\n\ndef _needs_water(tile: Any) -> bool:\n    return _is_plant(tile) and not bool(tile.get("watered_today", False))\n\n\ndef _manhattan(a: Coord, b: Coord) -> int:\n    return abs(a[0] - b[0]) + abs(a[1] - b[1])\n\n\ndef _nearest(\n    origin: Coord,\n    candidates: Iterable[Coord],\n    reserved: Set[Coord],\n) -> Optional[Coord]:\n    available = [p for p in candidates if p not in reserved]\n    if not available:\n        return None\n    return min(available, key=lambda p: (_manhattan(origin, p), p[1], p[0]))\n\n\ndef _scan_tasks(\n    tiles: Sequence[Sequence[Any]],\n    day: int,\n) -> Dict[str, List[Coord]]:\n    tasks = {"harvest": [], "water": [], "weed": [], "plant": []}\n    for y, row in enumerate(tiles):\n        for x, tile in enumerate(row):\n            pos = (x, y)\n            if tile == "LOCKED":\n                continue\n            if _harvest_ready(tile, day):\n                tasks["harvest"].append(pos)\n            elif _needs_water(tile):\n                tasks["water"].append(pos)\n            elif _is_weed(tile):\n                tasks["weed"].append(pos)\n            elif _is_unlocked_empty(tile):\n                tasks["plant"].append(pos)\n    return tasks\n\n\ndef _target_crop(day: int, money: float) -> Optional[str]:\n    if day <= CROP_RULES["MELON"]["last_plant_day"] and money >= MELON_START_CASH:\n        return "MELON"\n    if day <= CROP_RULES["WHEAT"]["last_plant_day"]:\n        return "WHEAT"\n    return None\n\n\ndef _unit_action(\n    pos: Coord,\n    tiles: Sequence[Sequence[Any]],\n    tasks: Dict[str, List[Coord]],\n    day: int,\n    crop_to_plant: Optional[str],\n    seeds_left: Dict[str, int],\n    reserved: Set[Coord],\n) -> Action:\n    x, y = pos\n    tile = tiles[y][x]\n\n    # Prefer productive work on the current square before moving.\n    if _harvest_ready(tile, day):\n        reserved.add(pos)\n        return ["HARVEST"]\n\n    if _needs_water(tile):\n        reserved.add(pos)\n        return ["WATER"]\n\n    if _is_weed(tile):\n        reserved.add(pos)\n        return ["DIG"]\n\n    if (\n        tile is None\n        and crop_to_plant\n        and seeds_left.get(crop_to_plant, 0) > 0\n    ):\n        seeds_left[crop_to_plant] -= 1\n        reserved.add(pos)\n        return ["PLANT", crop_to_plant]\n\n    # Route to the highest-priority unreserved task.\n    for task_name in ("harvest", "water", "weed"):\n        target = _nearest(pos, tasks[task_name], reserved)\n        if target is not None:\n            reserved.add(target)\n            step = _step_toward(pos, target)\n            return [step] if step else ["PASS"]\n\n    if crop_to_plant and seeds_left.get(crop_to_plant, 0) > 0:\n        target = _nearest(pos, tasks["plant"], reserved)\n        if target is not None:\n            reserved.add(target)\n            step = _step_toward(pos, target)\n            return [step] if step else ["PASS"]\n\n    return ["PASS"]\n\n\ndef _market_orders(\n    obs: Dict[str, Any],\n    farm: Dict[str, Any],\n    private: Dict[str, Any],\n    crop_to_plant: Optional[str],\n    active_workers: int,\n) -> List[Action]:\n    day = int(obs.get("day", 0) or 0)\n    hour = int(obs.get("hour", 0) or 0)\n    money = float(farm.get("money", 0) or 0)\n    seeds = private.get("seeds") or {}\n    shed = private.get("shed") or {}\n    prices = _prices(obs)\n\n    orders: List[Action] = []\n\n    # Sell stocked crops on acceptable prices, and force liquidation near season end.\n    for crop, rule in CROP_RULES.items():\n        qty = int(shed.get(crop, 0) or 0)\n        if qty <= 0:\n            continue\n        price = float(prices.get(crop, rule["base_price"]) or 0)\n        threshold = float(rule["base_price"]) * SELL_PRICE_RATIO\n        if day >= LATE_LIQUIDATION_DAY or price >= threshold:\n            orders.append(["SELL", crop, qty])\n\n    # Seed purchases are a buffer for future turns; planting only consumes seeds\n    # already visible in private state, avoiding same-turn ordering assumptions.\n    if crop_to_plant:\n        have = int(seeds.get(crop_to_plant, 0) or 0)\n        desired = max(TARGET_SEED_BUFFER, active_workers * 2)\n        need = max(0, desired - have)\n        cost = int(CROP_RULES[crop_to_plant]["seed_cost"])\n        affordable = max(0, int(money // cost))\n        buy = min(need, affordable)\n        if buy > 0:\n            orders.append(["BUY_SEED", crop_to_plant, buy])\n\n    # Hire only once at the start of each in-game day. Hires are deliberately\n    # capped until local evaluation proves that more labor pays for itself.\n    if hour == 0 and day < LATE_LIQUIDATION_DAY:\n        active_plants = sum(\n            1\n            for row in (farm.get("tiles") or [])\n            for tile in row\n            if _is_plant(tile)\n        )\n        target_hands = min(MAX_DAILY_HIRES, max(0, active_plants // 5))\n        existing_hands = len(farm.get("hands") or [])\n        hires = max(0, target_hands - existing_hands)\n        orders.extend([["HIRE"] for _ in range(hires)])\n\n    return orders[:10]\n\n\ndef agent(obs: Dict[str, Any]) -> Dict[str, Any]:\n    """Kaggle entrypoint: return farmer, hands, and market actions."""\n    farms = obs.get("farms") or []\n    player = int(obs.get("player", 0) or 0)\n\n    if player < 0 or player >= len(farms):\n        return {"farmer": ["PASS"], "hands": [], "market": []}\n\n    farm = farms[player]\n    tiles = farm.get("tiles") or []\n    if not tiles:\n        return {"farmer": ["PASS"], "hands": [], "market": []}\n\n    private = obs.get("private") or {}\n    seeds = dict(private.get("seeds") or {})\n    day = int(obs.get("day", 0) or 0)\n    money = float(farm.get("money", 0) or 0)\n    crop_to_plant = _target_crop(day, money)\n\n    tasks = _scan_tasks(tiles, day)\n    reserved: Set[Coord] = set()\n\n    farmer_pos = tuple(farm.get("farmer") or (0, 0))\n    hand_positions = [tuple(p) for p in (farm.get("hands") or [])]\n\n    farmer_action = _unit_action(\n        farmer_pos,\n        tiles,\n        tasks,\n        day,\n        crop_to_plant,\n        seeds,\n        reserved,\n    )\n\n    hand_actions = [\n        _unit_action(\n            pos,\n            tiles,\n            tasks,\n            day,\n            crop_to_plant,\n            seeds,\n            reserved,\n        )\n        for pos in hand_positions\n    ]\n\n    market_orders = _market_orders(\n        obs,\n        farm,\n        private,\n        crop_to_plant,\n        1 + len(hand_positions),\n    )\n\n    return {\n        "farmer": farmer_action,\n        "hands": hand_actions,\n        "market": market_orders,\n    }\n\n\n# Useful aliases for notebooks that expect a differently named callable.\nmain = agent\n', 'scripts/experiment_eval.py': '"""Paired evaluation and exact official-interpreter telemetry for own candidates."""\n\nimport argparse\nfrom collections import Counter, defaultdict\nfrom copy import deepcopy\nfrom datetime import datetime, timezone\nimport hashlib\nimport importlib.util\nimport json\nfrom pathlib import Path\nimport statistics\nimport time\n\nimport numpy as np\nfrom kaggle_environments import __version__, make\nfrom kaggle_environments.envs.kaggriculture import kaggriculture as engine\n\nROOT = Path(__file__).resolve().parents[1]\n\n\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef load_agent(path, disabled=()):\n    spec = importlib.util.spec_from_file_location("isolated_agent", Path(path))\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    for flag in disabled:\n        if not hasattr(module, flag):\n            raise ValueError("Unknown ablation flag")\n        setattr(module, flag, False)\n    fn = module.agent\n\n    def call(obs, config):\n        return fn(obs, config) if fn.__code__.co_argcount >= 2 else fn(obs)\n\n    return call\n\n\ndef run_game(candidate, opponent, seed, seat, disabled=()):\n    policy = load_agent(candidate, disabled)\n    rival = opponent if opponent in {"starter", "pass"} else load_agent(opponent)\n    env = make("kaggriculture", configuration={"seed": seed, "episodeSteps": 720})\n    counters = Counter()\n    actions = Counter()\n    latencies = []\n    current = []\n    original_interpreter = env.interpreter\n    originals = {n: getattr(engine, n) for n in (\n        "_apply_unit_action", "_commit_unit", "_do_hire", "_do_buy_land", "_drop_inventories_to_shed")}\n    fills = Counter()\n\n    def measured(obs, config):\n        begin = time.perf_counter()\n        try:\n            result = policy(obs, config)\n            json.dumps(result, allow_nan=False)\n            if not isinstance(result, dict) or any(not isinstance(result.get(k), list) for k in ("farmer", "hands", "market")):\n                counters["contract_errors"] += 1\n            elif len(result["hands"]) != len(obs["farms"][seat]["hands"]) or len(result["market"]) > 10:\n                counters["contract_errors"] += 1\n            return result\n        except Exception:\n            counters["exceptions"] += 1\n            raise\n        finally:\n            latencies.append(time.perf_counter() - begin)\n\n    def is_ours(farm):\n        return bool(current) and farm is current[0].observation.farms[seat]\n\n    def stock(private):\n        return sum(private["shed"].values()) + sum(sum(x.values()) for x in private["inventories"])\n\n    def unit(farm, private, idx, action, board, day, tpd, capacity=100):\n        ours = is_ours(farm)\n        before = deepcopy((farm, private)) if ours else None\n        answer = originals["_apply_unit_action"](farm, private, idx, action, board, day, tpd, capacity)\n        if ours:\n            op = action[0] if isinstance(action, list) and action else "MALFORMED"\n            actions[op] += 1\n            if op != "PASS" and before == (farm, private):\n                counters["unit_noops"] += 1\n            if op == "DROP":\n                counters["overflow_units"] += max(0, stock(before[1]) - stock(private))\n        return answer\n\n    def commit(op, item, price, farm, private, market, capacity=100):\n        result = originals["_commit_unit"](op, item, price, farm, private, market, capacity)\n        if is_ours(farm):\n            if result:\n                fills[(op, item)] += 1\n            else:\n                counters["failed_market_attempts"] += 1\n        return result\n\n    def hire(farm, private, board, mult=1):\n        before = len(farm["hands"])\n        originals["_do_hire"](farm, private, board, mult)\n        if is_ours(farm):\n            fills[("HIRE", "")] += len(farm["hands"]) - before\n\n    def land(farm, board):\n        before = len(farm["unlocked_quadrants"])\n        originals["_do_buy_land"](farm, board)\n        if is_ours(farm):\n            fills[("BUY_LAND", "")] += len(farm["unlocked_quadrants"]) - before\n\n    def drop(private, capacity):\n        ours = bool(current) and private is current[seat].observation.private\n        before = stock(private) if ours else 0\n        originals["_drop_inventories_to_shed"](private, capacity)\n        if ours:\n            counters["overflow_units"] += max(0, before - stock(private))\n\n    def interpreter(state, environment):\n        nonlocal current\n        current = state\n        if not state[0].observation.get("farms"):\n            return original_interpreter(state, environment)\n        fills.clear()\n        a = state[seat].action or {}\n        requested = Counter()\n        for order in a.get("market", []):\n            parsed = engine._parse_order(order)\n            if parsed is None:\n                counters["contract_errors"] += 1\n            elif parsed["type"] in {"HIRE", "BUY_LAND"}:\n                requested[(parsed["type"], "")] += 1\n            else:\n                requested[(parsed["type"], parsed["item"])] += parsed["remaining"]\n        plant = Counter(x[1] for x in [a.get("farmer", []), *a.get("hands", [])] if len(x) >= 2 and x[0] == "PLANT")\n        seeds = state[seat].observation.private["seeds"]\n        counters["atomic_plant_rejections"] += sum(n for crop, n in plant.items() if n > seeds.get(crop, 0))\n        result = original_interpreter(state, environment)\n        counters["unfilled_requested_market_units"] += sum(max(0, n - fills[k]) for k, n in requested.items())\n        return result\n\n    engine._apply_unit_action, engine._commit_unit = unit, commit\n    engine._do_hire, engine._do_buy_land = hire, land\n    engine._drop_inventories_to_shed = drop\n    env.interpreter = interpreter\n    begin = time.perf_counter()\n    try:\n        players = [rival, rival]\n        players[seat] = measured\n        env.run(players)\n    finally:\n        for name, fn in originals.items():\n            setattr(engine, name, fn)\n    final = env.steps[-1]\n    valid = all(s.status == "DONE" and isinstance(s.reward, (int, float)) for s in final) and len(env.steps) == 720\n    ours, theirs = final[seat].reward, final[1 - seat].reward\n    private = final[seat].observation.private\n    row = {"seed": seed, "seat": seat, "opponent": Path(opponent).stem,\n           "ours": ours, "theirs": theirs, "statuses": [s.status for s in final],\n           "valid_terminal": valid, "faults": dict(counters), "actions": dict(actions),\n           "calls": len(latencies), "runtime_mean": statistics.fmean(latencies) if latencies else None,\n           "runtime_p95": float(np.quantile(latencies, .95)) if latencies else None,\n           "runtime_p99": float(np.quantile(latencies, .99)) if latencies else None,\n           "runtime_max": max(latencies, default=None), "wall_seconds": time.perf_counter() - begin,\n           "terminal_carried_units": sum(sum(x.values()) for x in private["inventories"]),\n           "terminal_shed_units": sum(private["shed"].values())}\n    if valid:\n        row["margin"] = ours - theirs\n        row["points"] = 1. if ours > theirs else .5 if ours == theirs else 0.\n    return row\n\n\ndef summarize(rows):\n    groups = defaultdict(list)\n    for row in rows:\n        groups[row["opponent"]].append(row)\n    results = {}\n    for name, group in groups.items():\n        valid = all(r["valid_terminal"] for r in group)\n        errors = Counter()\n        for r in group:\n            errors.update(r["faults"])\n        results[name] = {"games": len(group), "valid": valid, "faults": dict(errors)}\n        if valid:\n            margins = [r["margin"] for r in group]\n            results[name].update(points=statistics.fmean(r["points"] for r in group),\n                                 wins=sum(r["points"] == 1 for r in group),\n                                 draws=sum(r["points"] == .5 for r in group),\n                                 mean_bank=statistics.fmean(r["ours"] for r in group),\n                                 mean_margin=statistics.fmean(margins),\n                                 median_margin=statistics.median(margins),\n                                 std_margin=statistics.stdev(margins) if len(margins) > 1 else 0.)\n    return results\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--agent", type=Path, required=True)\n    parser.add_argument("--opponents", nargs="+", required=True)\n    parser.add_argument("--seed-start", type=int, required=True)\n    parser.add_argument("--seeds", type=int, required=True)\n    parser.add_argument("--disable", nargs="*", default=[])\n    parser.add_argument("--output", type=Path, required=True)\n    args = parser.parse_args()\n    if args.seeds < 1 or args.output.exists():\n        raise SystemExit("Invalid count or output already exists")\n    rows = []\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    progress = args.output.with_suffix(".progress.jsonl")\n    with progress.open("x") as log:\n        for opponent in args.opponents:\n            for seed in range(args.seed_start, args.seed_start + args.seeds):\n                for seat in (0, 1):\n                    row = run_game(args.agent, opponent, seed, seat, args.disable)\n                    rows.append(row)\n                    log.write(json.dumps(row) + "\\n")\n                    log.flush()\n                    print(json.dumps({k: row[k] for k in ("seed", "seat", "opponent", "ours", "theirs", "faults")}), flush=True)\n    report = {"timestamp": datetime.now(timezone.utc).isoformat(), "agent_sha": sha(args.agent),\n              "engine_version": __version__, "engine_sha": sha(engine.__file__),\n              "evaluator_sha": sha(__file__), "disabled": args.disable,\n              "opponent_hashes": {Path(p).stem: sha(p) for p in args.opponents if Path(p).is_file()},\n              "seed_start": args.seed_start, "seed_count": args.seeds, "seat_swapped": True,\n              "summary": summarize(rows), "rows": rows, "submission_approval": False}\n    with args.output.open("x") as handle:\n        json.dump(report, handle, indent=2)\n        handle.write("\\n")\n    print(json.dumps(report["summary"], indent=2))\n    if not all(r["valid_terminal"] for r in rows):\n        raise SystemExit(1)\n\n\nif __name__ == "__main__":\n    main()\n', 'tests/test_independent_controller.py': 'import importlib.util\nimport json\nfrom pathlib import Path\n\nfrom kaggle_environments.envs.kaggriculture import kaggriculture as engine\n\nPATH = Path(__file__).resolve().parents[1] / "agents/candidates/exp006_independent.py"\nspec = importlib.util.spec_from_file_location("independent", PATH)\npolicy = importlib.util.module_from_spec(spec)\nspec.loader.exec_module(policy)\n\n\ndef observation(day=0, hour=0, money=3000, hands=()):\n    farm = engine._new_farm(10, money)\n    farm["hands"] = list(hands)\n    private = engine._new_private()\n    private["inventories"] += [{} for _ in hands]\n    return {"player": 0, "day": day, "hour": hour,\n            "farms": [farm, engine._new_farm(10, money)], "private": private,\n            "market": engine._new_market(), "town": {"unlocked_shops": []}}\n\n\ndef test_price_function_matches_official_integer_quotes():\n    for item in engine.PRODUCTS:\n        for inventory in range(9000, 11001, 7):\n            assert policy.price(item, inventory) == engine.market_price(item, inventory)\n\n\ndef test_sparse_price_overrides_match_official():\n    params = engine._resolve_market_params({"MILK": {"above_target": .5}, "CARROT": {"below_func": "linear"}})\n    for item in params:\n        for inventory in (9000, 9999, 10000, 10001, 10100):\n            assert policy.price(item, inventory, params) == engine.market_price(item, inventory, params)\n\n\ndef test_fallback_contract():\n    assert policy.agent({}) == {"farmer": ["PASS"], "hands": [], "market": []}\n\n\ndef test_new_crop_is_watered_before_night():\n    obs = observation(hour=23)\n    obs["farms"][0]["tiles"][4][4] = engine._new_plant("MELON", 0, 24)\n    assert policy.agent(obs)["farmer"] == ["WATER"]\n\n\ndef test_no_last_hour_planting():\n    obs = observation(hour=23)\n    obs["private"]["seeds"]["MELON"] = 10\n    assert policy.agent(obs)["farmer"][0] != "PLANT"\n\n\ndef test_shared_square_work_is_not_duplicated():\n    obs = observation(hands=[[4, 4]])\n    obs["farms"][0]["tiles"][4][4] = engine._new_plant("MELON", 0, 24)\n    action = policy.agent(obs)\n    assert action["farmer"] == ["WATER"]\n    assert action["hands"][0] != ["WATER"]\n\n\ndef test_visible_seed_budget():\n    obs = observation(hands=[[3, 4], [2, 4], [1, 4]])\n    obs["private"]["seeds"]["MELON"] = 1\n    action = policy.agent(obs)\n    assert sum(a[0] == "PLANT" for a in [action["farmer"], *action["hands"]]) <= 1\n\n\ndef test_peak_day_water_precedes_harvest():\n    obs = observation(day=4)\n    tile = engine._new_plant("WHEAT", 0, 24)\n    tile["yield_units"] = 3\n    obs["farms"][0]["tiles"][4][4] = tile\n    assert policy.agent(obs)["farmer"] == ["WATER"]\n    tile["watered_today"] = True\n    tile["yield_units"] = 4\n    assert policy.agent(obs)["farmer"] == ["HARVEST"]\n\n\ndef test_final_turn_drop_can_sell_same_turn():\n    obs = observation(day=29, hour=22)\n    obs["private"]["inventories"][0] = {"MELON": 8}\n    action = policy.agent(obs)\n    assert action["farmer"] == ["DROP"]\n    assert ["SELL", "MELON", 8] in action["market"]\n\n\ndef test_drop_will_not_discard_overflow():\n    obs = observation(day=29, hour=22)\n    obs["private"]["shed"]["WHEAT"] = 99\n    obs["private"]["inventories"][0] = {"MELON": 8}\n    action = policy.agent(obs)\n    assert action["farmer"] == ["PLACE", "MELON", 1]\n\n\ndef test_purchases_do_not_borrow_unknown_sale_receipts():\n    for money in (0, 10, 79, 80, 250, 500, 3000):\n        obs = observation(money=money)\n        obs["private"]["shed"]["MELON"] = 100\n        action = policy.agent(obs)\n        spent, hires, land = 0, 0, 0\n        for order in action["market"]:\n            if order[0] == "HIRE":\n                spent += engine._hire_cost(hires)\n                hires += 1\n            elif order[0] == "BUY_LAND":\n                spent += engine.LAND_PRICES[land]\n                land += 1\n            elif order[0] == "BUY_SEED":\n                spent += engine.CROPS[order[1]]["seed"] * order[2]\n        assert spent <= money\n        assert len(action["market"]) <= 10\n        json.dumps(action, allow_nan=False)\n', 'LICENSE.txt': '                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright [yyyy] [name of copyright owner]\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n', 'NOTICE.txt': "EXP-006 Independent Crop Controller\nRepository owner: whzy3185\nKaggle account: muelsyse111\n\nCandidate policy, working-capital budget, task assignment and terminal transport\nwere independently implemented in this repository with AI assistance.\nNo public opponent's source, route tapes or model weights are included.\n\nGame rules, constants and price equations are documented by Kaggle's\nkaggle-environments project, licensed under Apache License 2.0:\nhttps://github.com/Kaggle/kaggle-environments/tree/master/kaggle_environments/envs/kaggriculture\nInspected package version: 1.32.7.\n\nPublic strategy research informed the questions and evaluation design; it is not\nclaimed as our original research. Source-specific provenance is recorded in the\nrepository research directory. Shape the Shop v11 by tetsutani is an attributed\nevaluation reference only and is not part of this competition artifact.\n"}
EXPECTED_AGENT_SHA = '84f11eaeceb1f2053c51947d7369b71ef6c7bdebfc66b815fbc0d04d62ecbd77'
for name, source in FILES.items():
    path = pathlib.Path(name)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source)
assert hashlib.sha256(pathlib.Path("main.py").read_bytes()).hexdigest() == EXPECTED_AGENT_SHA
print("Candidate source hash verified:", EXPECTED_AGENT_SHA)


In [ ]:
import xml.etree.ElementTree as ET
run = subprocess.run([sys.executable, "-m", "pytest", "tests/test_independent_controller.py", "-q", "--junitxml=cloud-tests.xml"], capture_output=True, text=True)
print(run.stdout)
assert run.returncode == 0, "Cloud unit/contract tests failed"
suites = ET.parse("cloud-tests.xml").getroot().findall("testsuite")
test_count = sum(int(s.attrib["tests"]) for s in suites)
assert test_count >= 11


In [ ]:
run = subprocess.run([sys.executable, "scripts/experiment_eval.py", "--agent", "main.py", "--opponents", "legacy_parent.py", "main.py", "--seed-start", "4001", "--seeds", "2", "--output", "cloud-runtime.json"], capture_output=True, text=True)
assert run.returncode == 0, "Cloud evaluation failed; inspect cloud-runtime.progress.jsonl"
report = json.loads(pathlib.Path("cloud-runtime.json").read_text())
assert report["agent_sha"] == EXPECTED_AGENT_SHA
assert len(report["rows"]) == 8
for row in report["rows"]:
    assert row["valid_terminal"] and row["calls"] == 719
    assert not any(row["faults"].values()), row["faults"]
    assert row["runtime_max"] < 0.5
print(json.dumps(report["summary"], indent=2))


In [ ]:
import gzip, io, tarfile
buffer = io.BytesIO()
with gzip.GzipFile(fileobj=buffer, mode="wb", filename="", mtime=0) as gz:
    with tarfile.open(fileobj=gz, mode="w", format=tarfile.USTAR_FORMAT) as tar:
        for name in ("main.py", "LICENSE.txt", "NOTICE.txt"):
            payload = pathlib.Path(name).read_bytes()
            entry = tarfile.TarInfo(name)
            entry.size, entry.mode, entry.mtime = len(payload), 0o644, 0
            tar.addfile(entry, io.BytesIO(payload))
artifact = buffer.getvalue()
pathlib.Path("submission.tar.gz").write_bytes(artifact)
with tarfile.open("submission.tar.gz") as tar:
    assert tar.getnames() == ["main.py", "LICENSE.txt", "NOTICE.txt"]
    assert hashlib.sha256(tar.extractfile("main.py").read()).hexdigest() == EXPECTED_AGENT_SHA
verification = {
    "exp_id": "EXP-006", "agent_sha": EXPECTED_AGENT_SHA,
    "artifact_sha": hashlib.sha256(artifact).hexdigest(),
    "engine_version": report["engine_version"], "tests": test_count,
    "episodes": len(report["rows"]), "faults": 0,
    "runtime_max": max(r["runtime_max"] for r in report["rows"]),
    "scope": "Runtime and baseline validation, not evidence of beating strong public agents",
}
pathlib.Path("runtime-verification.json").write_text(json.dumps(verification, indent=2))
print(json.dumps(verification, indent=2))
